# **Imports & Configs**

In [1]:
import importlib, subprocess, sys

for pkg in ['wandb', 'transformers']:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('Dependencies ready.')

Dependencies ready.


In [2]:
import os
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
import wandb
from kaggle_secrets import UserSecretsClient


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

#  PATHS 
BASE_DIR    = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
STEMS_DIR   = os.path.join(BASE_DIR, 'genres_stems')
MASHUPS_DIR = os.path.join(BASE_DIR, 'mashups')
ESC_DIR     = os.path.join(BASE_DIR, 'ESC-50-master', 'audio')
TEST_CSV    = os.path.join(BASE_DIR, 'test.csv')

# LABEL MAPPINGS
GENRES    = sorted(['blues','classical','country','disco','hiphop','jazz','metal','pop','reggae','rock'])
GENRE2IDX = {g: i for i, g in enumerate(GENRES)}
IDX2GENRE = {i: g for g, i in GENRE2IDX.items()}

# WANDB 
# secrets   = UserSecretsClient()
# wandb_key = secrets.get_secret("wandb_api_key")
# wandb.login(key=wandb_key)

print(f'Genres: {GENRES}')
print('Setup complete ✅')

Device: cuda
Genres: ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']
Setup complete ✅


In [3]:
 # import wandb 
 # wandb.login()

# **Preprocessing**

In [4]:
def extract_melspectrogram_multicrop(file_path, sr=22050, n_mels=128, max_frames=512):
    
    y, sr = librosa.load(file_path, sr=sr)

    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels, hop_length=512)
    
    # Convert power → decibel scale (matches human loudness perception)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    
    total_frames = mel_db.shape[1]
    
    if total_frames <= max_frames:
        # Audio too short → pad with zeros and return same crop 3 times
        padded = np.pad(mel_db, ((0,0),(0, max_frames - total_frames)))
        padded = (padded - padded.min()) / (padded.max() - padded.min() + 1e-6)
        return [padded, padded, padded]
    
    # Extract 3 crops — start, center, end — 3x free data
    start_crop  = mel_db[:, :max_frames]
    mid         = (total_frames - max_frames) // 2
    center_crop = mel_db[:, mid:mid + max_frames]
    end_crop    = mel_db[:, total_frames - max_frames:]
    
    crops = []
    for crop in [start_crop, center_crop, end_crop]:
        # Normalize each crop independently to [0, 1]
        crop = (crop - crop.min()) / (crop.max() - crop.min() + 1e-6)
        crops.append(crop)
    
    return crops  # list of 3 arrays each (128, 512)


def extract_melspectrogram_from_waveform_multicrop(y, sr=22050, n_mels=128, max_frames=512):
    # Same as above but takes waveform directly — avoids disk read for synthetic mashups
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels, hop_length=512)
    
    # Convert power → decibel scale
    mel_db = librosa.power_to_db(mel, ref=np.max)
    
    total_frames = mel_db.shape[1]
    
    if total_frames <= max_frames:
        padded = np.pad(mel_db, ((0,0),(0, max_frames - total_frames)))
        padded = (padded - padded.min()) / (padded.max() - padded.min() + 1e-6)
        return [padded, padded, padded]
    
    start_crop  = mel_db[:, :max_frames]
    mid         = (total_frames - max_frames) // 2
    center_crop = mel_db[:, mid:mid + max_frames]
    end_crop    = mel_db[:, total_frames - max_frames:]
    
    crops = []
    for crop in [start_crop, center_crop, end_crop]:
        crop = (crop - crop.min()) / (crop.max() - crop.min() + 1e-6)
        crops.append(crop)
    
    return crops  # list of 3 arrays each (128, 512)


def create_synthetic_mashup(genre, sr=22050):
    genre_path     = os.path.join(STEMS_DIR, genre)
    songs          = os.listdir(genre_path)
    selected_songs = random.sample(songs, 4)  # pick 4 unique songs from same genre
    stems          = ['drums', 'vocals', 'bass', 'other']
    waveforms      = []

    for song, stem in zip(selected_songs, stems):
        path = os.path.join(genre_path, song, f'{stem}.wav')
        y, _ = librosa.load(path, sr=sr)
        
        # Tempo augmentation — mimic test mashup tempo syncing (70% probability)
        if random.random() < 0.7:
            rate = random.uniform(0.95, 1.05)  # ±5% speed, pitch preserved
            y    = librosa.effects.time_stretch(y, rate=rate)
        
        waveforms.append(y)

    # Pad all stems to same length then mix by summing
    max_len   = max(len(w) for w in waveforms)
    waveforms = [np.pad(w, (0, max_len - len(w))) for w in waveforms]
    mixed     = np.sum(waveforms, axis=0)
    
    # Normalize to [-1, 1] to prevent clipping
    mixed = mixed / (np.max(np.abs(mixed)) + 1e-6)
    return mixed


def add_noise(waveform, sr=22050):
    noise_files     = os.listdir(ESC_DIR)
    num_noises      = random.randint(1, 4)            # inject 1-4 random noise clips
    selected_noises = random.sample(noise_files, num_noises)
    augmented       = waveform.copy()

    for noise_file in selected_noises:
        noise, _ = librosa.load(os.path.join(ESC_DIR, noise_file), sr=sr)
        if len(noise) > len(augmented):
            noise = noise[:len(augmented)]             # crop noise if longer than audio
        offset    = random.randint(0, len(augmented) - len(noise))  # random position
        intensity = random.uniform(0.05, 0.3)          # random intensity
        augmented[offset:offset + len(noise)] += intensity * noise

    # Normalize after noise injection
    augmented = augmented / (np.max(np.abs(augmented)) + 1e-6)
    return augmented

print("All functions defined ✅")

All functions defined ✅


# **Building Training dataset**

In [5]:
X = []
y = []

for genre in tqdm(GENRES, desc='Processing genres'):
    genre_path = os.path.join(STEMS_DIR, genre)
    songs      = os.listdir(genre_path)

    # Part A: Individual stems — 3 crops each
    for song in songs:
        for stem in ['drums', 'vocals', 'bass', 'other']:
            path  = os.path.join(genre_path, song, f'{stem}.wav')
            crops = extract_melspectrogram_multicrop(path)
            for crop in crops:
                X.append(crop)
                y.append(GENRE2IDX[genre])

    # Part B: 500 synthetic mashups — 3 crops each
    for _ in range(500):
        waveform = create_synthetic_mashup(genre)
        waveform = add_noise(waveform)
        crops    = extract_melspectrogram_from_waveform_multicrop(waveform)
        for crop in crops:
            X.append(crop)
            y.append(GENRE2IDX[genre])

# Fix shape inconsistencies
X = np.array([x[:, :512] if x.shape[1] >= 512 else
              np.pad(x, ((0,0),(0, 512 - x.shape[1]))) for x in X])
y = np.array(y)

# Save to avoid rerunning next session
np.save('/kaggle/working/X_train_data.npy', X)
np.save('/kaggle/working/y_train_data.npy', y)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Min: {X.min():.4f}, Max: {X.max():.4f}")
print(f"Samples per genre: {np.bincount(y)}")

Processing genres: 100%|██████████| 10/10 [1:21:06<00:00, 486.69s/it]


X shape: (27000, 128, 512)
y shape: (27000,)
Min: 0.0000, Max: 1.0000
Samples per genre: [2700 2700 2700 2700 2700 2700 2700 2700 2700 2700]


# **Enet dataloader**

In [6]:
# import timm

# # LOAD PROCESSED DATA
# specs = np.load('/kaggle/working/X_train_data.npy', mmap_mode='r')
# labels = np.load('/kaggle/working/y_train_data.npy')
# print(f"Spectrograms: {specs.shape}, Labels: {labels.shape}")

# class SpectrogramDataset(Dataset):
#     def __init__(self, specs, labels, training=False):
#         self.specs    = specs
#         self.labels   = torch.tensor(labels, dtype=torch.long)
#         self.training = training

#     def __len__(self):
#         return len(self.specs)

#     def augment(self, spec):
#         # Frequency masking (zero out random mel bands)
#         f_width = random.randint(0, 20)
#         f_start = random.randint(0, spec.shape[0] - f_width)
#         spec[f_start:f_start + f_width, :] = 0.0
#         # Time masking — zero out random time frames
#         t_width = random.randint(0, 40)
#         t_start = random.randint(0, spec.shape[1] - t_width)
#         spec[:, t_start:t_start + t_width] = 0.0
#         return spec

#     def __getitem__(self, idx):
#         # Convert one sample at a time (avoids OOM)
#         spec = torch.tensor(self.specs[idx].copy(), dtype=torch.float32)
#         if self.training:
#             spec = self.augment(spec)
#         # EfficientNet needs 3 channel input (repeat grayscale spectrogram)
#         spec = spec.unsqueeze(0).expand(3, -1, -1).clone()
#         return spec, self.labels[idx]

# # Splitting by index
# idx        = np.arange(len(specs))
# train_idx, val_idx = train_test_split(
#     idx, test_size=0.2, random_state=42, stratify=labels)

# train_ds = SpectrogramDataset(specs[train_idx], labels[train_idx], training=True)
# val_ds   = SpectrogramDataset(specs[val_idx],   labels[val_idx],   training=False)

# train_dl = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
# val_dl   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

# sample_x, sample_y = next(iter(train_dl))
# print(f"Input shape : {sample_x.shape}")
# print(f"Label shape : {sample_y.shape}")

# **Enet training loop**

In [7]:
# from sklearn.metrics import f1_score, accuracy_score

# # Loading EfficientNet-B3 with ImageNet weights
# enet  = timm.create_model('efficientnet_b3', pretrained=True, num_classes=10)
# enet  = enet.to(DEVICE)
# print(f"Parameters: {sum(p.numel() for p in enet.parameters()):,}")

# loss_fn   = nn.CrossEntropyLoss()
# opt       = torch.optim.AdamW(enet.parameters(), lr=1e-4, weight_decay=1e-4)
# lr_sched  = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=3, factor=0.5)

# wandb.init(project='23f2004377-t12026', name="Enet-B3")

# def run_train_epoch(net, loader, opt, loss_fn):
#     net.train()
#     epoch_loss, n_correct = 0.0, 0
#     for xb, yb in loader:
#         xb, yb = xb.to(DEVICE), yb.to(DEVICE)
#         opt.zero_grad()
#         out  = net(xb)
#         loss = loss_fn(out, yb)
#         loss.backward()
#         opt.step()
#         epoch_loss += loss.item()
#         n_correct  += (out.argmax(1) == yb).sum().item()
#     return epoch_loss / len(loader), n_correct / len(loader.dataset)

# def run_val_epoch(net, loader, loss_fn):
#     net.eval()
#     epoch_loss = 0.0
#     preds_all, labels_all = [], []
#     with torch.no_grad():
#         for xb, yb in loader:
#             xb, yb = xb.to(DEVICE), yb.to(DEVICE)
#             out        = net(xb)
#             epoch_loss += loss_fn(out, yb).item()
#             preds_all.extend(out.argmax(1).cpu().tolist())
#             labels_all.extend(yb.cpu().tolist())
#     avg_loss = epoch_loss / len(loader)
#     acc      = accuracy_score(labels_all, preds_all)
#     macro_f1 = f1_score(labels_all, preds_all, average='macro')
#     return avg_loss, acc, macro_f1

# NUM_EPOCHS   = 20
# best_f1      = 0.0
# best_ckpt    = '/kaggle/working/enet_best.pth'

# for ep in range(1, NUM_EPOCHS + 1):
#     tr_loss, tr_acc        = run_train_epoch(enet, train_dl, opt, loss_fn)
#     vl_loss, vl_acc, vl_f1 = run_val_epoch(enet, val_dl, loss_fn)
#     lr_sched.step(vl_loss)

#     wandb.log({
#         'epoch': ep,
#         'train/loss': tr_loss, 'train/acc': tr_acc,
#         'val/loss': vl_loss,   'val/acc': vl_acc, 'val/f1': vl_f1
#     })

#     status = ''
#     if vl_f1 > best_f1:
#         best_f1 = vl_f1
#         torch.save(enet.state_dict(), best_ckpt)
#         status = ' ✅ saved'

#     print(f"[{ep:02d}/{NUM_EPOCHS}] "
#           f"tr_loss={tr_loss:.4f} tr_acc={tr_acc:.4f} | "
#           f"vl_loss={vl_loss:.4f} vl_acc={vl_acc:.4f} vl_f1={vl_f1:.4f}{status}")

# wandb.finish()
# print(f"\nBest Macro F1: {best_f1:.4f}")

# **Enet inference**

In [8]:
# # Reload best checkpoint for inference
# enet = timm.create_model('efficientnet_b3', pretrained=False, num_classes=10)
# enet.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
# enet = enet.to(DEVICE)
# enet.eval()

# test_df  = pd.read_csv(TEST_CSV)
# results  = []

# print("Running inference...")

# for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Predicting'):
#     audio_path = os.path.join(str(DATA_ROOT), row['filename'])
#     crops      = extract_melspectrogram_multicrop(audio_path)
#     scores     = np.zeros(10)

#     for crop in crops:
#         tensor = torch.tensor(crop, dtype=torch.float32)
#         tensor = tensor.unsqueeze(0).expand(3,-1,-1).clone()
#         tensor = tensor.unsqueeze(0).to(DEVICE)
#         with torch.no_grad():
#             # scores += torch.softmax(enet(tensor), dim=1).squeeze().cpu().numpy()

#     predicted_genre = ID2LABEL[int(np.argmax(scores))]
#     results.append({'id': row['id'], 'genre': predicted_genre})

# submission = pd.DataFrame(results)
# submission.to_csv('/kaggle/working/submission_enet.csv', index=False)
# print(f"\nSaved! {submission.shape}")
# print(submission['genre'].value_counts())

## **enet standalone inference**

In [9]:
# import os
# import numpy as np
# import pandas as pd
# import torch
# import torch.nn as nn
# import librosa
# import timm
# from tqdm import tqdm

# # CONFIG
# BASE_DIR   = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
# TEST_CSV   = os.path.join(BASE_DIR, 'test.csv')
# MODEL_PATH = '/kaggle/input/models/arsh1ya/trained/other/default/1/enet_best (1).pth'
# DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# GENRES     = sorted(['blues','classical','country','disco','hiphop','jazz','metal','pop','reggae','rock'])
# ID2LABEL   = {i: g for i, g in enumerate(GENRES)}

# # LOAD MODEL
# enet = timm.create_model('efficientnet_b3', pretrained=False, num_classes=10)
# enet.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
# enet = enet.to(DEVICE)
# enet.eval()
# print("EfficientNet-B3 loaded ✅")

# # FEATURE EXTRACTION 
# def extract_crops(file_path, sr=22050, n_mels=128, max_frames=512):
#     y, sr        = librosa.load(file_path, sr=sr)
#     mel          = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels, hop_length=512)
#     mel_db       = librosa.power_to_db(mel, ref=np.max)
#     total_frames = mel_db.shape[1]
#     if total_frames <= max_frames:
#         padded = np.pad(mel_db, ((0,0),(0, max_frames - total_frames)))
#         padded = (padded - padded.min()) / (padded.max() - padded.min() + 1e-6)
#         return [padded, padded, padded]
#     start_crop  = mel_db[:, :max_frames]
#     mid         = (total_frames - max_frames) // 2
#     center_crop = mel_db[:, mid:mid + max_frames]
#     end_crop    = mel_db[:, total_frames - max_frames:]
#     crops = []
#     for crop in [start_crop, center_crop, end_crop]:
#         crop = (crop - crop.min()) / (crop.max() - crop.min() + 1e-6)
#         crops.append(crop)
#     return crops

# #  INFERENCE 
# test_df = pd.read_csv(TEST_CSV)
# results = []

# print("="*50)
# print("Inferencing starts...")
# print("="*50)

# for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
#     file_path = os.path.join(BASE_DIR, row['filename'])
#     crops     = extract_crops(file_path)
#     scores    = np.zeros(10)

#     for crop in crops:
#         tensor = torch.tensor(crop, dtype=torch.float32)
#         tensor = tensor.unsqueeze(0).expand(3,-1,-1).clone()
#         tensor = tensor.unsqueeze(0).to(DEVICE)
#         with torch.no_grad():
#             scores += torch.softmax(enet(tensor), dim=1).squeeze().cpu().numpy()

#     results.append({'id': row['id'], 'genre': ID2LABEL[int(np.argmax(scores))]})

# # SUBMISSION 
# submission = pd.DataFrame(results)
# submission.to_csv('submission.csv', index=False)
# print(f"\nDone! Shape: {submission.shape}")
# print(submission['genre'].value_counts())